# EEG Bandpass Filtering Demo

This notebook demonstrates loading EEG data for a single subject and run, extracting the raw signal, applying a bandpass filter (0.1-20 Hz), and plotting the results before and after filtering.

## Context
- Data from distractor decoding experiment
- EEG sampled at 512 Hz
- Bandpass filter: 0.1-20 Hz (as per preprocessing pipeline)

In [1]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import mne
from scipy.signal import butter, filtfilt


In [2]:
# Function to load EEG data from .gdf file
def load_eeg_data(gdf_path):
    """
    Load EEG data from a .gdf file.
    
    Parameters:
    gdf_path (str): Path to the .gdf file
    
    Returns:
    raw_data (np.ndarray): EEG signal matrix (n_samples x n_channels)
    sfreq (float): Sampling frequency
    ch_names (list): Channel names
    """
    raw = mne.io.read_raw_gdf(gdf_path, preload=True, verbose=False)
    raw_data = raw.get_data()  # Shape: (n_channels, n_samples)
    raw_data = raw_data.T  # Transpose to (n_samples, n_channels) for consistency
    sfreq = raw.info['sfreq']
    ch_names = raw.ch_names
    return raw_data, sfreq, ch_names

In [3]:
# Function to apply bandpass filter
def apply_bandpass_filter(data, sfreq, low_freq=0.1, high_freq=20.0, order=4):
    """
    Apply bandpass filter to EEG data.
    
    Parameters:
    data (np.ndarray): EEG signal (n_samples x n_channels)
    sfreq (float): Sampling frequency
    low_freq (float): Low cutoff frequency
    high_freq (float): High cutoff frequency
    order (int): Filter order
    
    Returns:
    filtered_data (np.ndarray): Filtered EEG signal
    """
    nyquist = sfreq / 2
    low = low_freq / nyquist
    high = high_freq / nyquist
    b, a = butter(order, [low, high], btype='band')
    
    # Apply filter to each channel
    filtered_data = np.zeros_like(data)
    for ch in range(data.shape[1]):
        filtered_data[:, ch] = filtfilt(b, a, data[:, ch])
    
    return filtered_data

In [ ]:
# Load EEG data for one subject and run
# Path structure: Box/CNBI/Attention_distraction/project_healthy/{subject}/{subject_date}/{subject_datetime_task}/{subject_datetime_task}.gdf
# Example: Box/CNBI/Attention_distraction/project_healthy/e38/e38_20260303/e38_20260303092652_stroop/e38_20260303092652_stroop.gdf
gdf_path = 'Box/CNBI/Attention_distraction/project_healthy/e38/e38_20260303/e38_20260303092652_stroop/e38_20260303092652_stroop.gdf'  # Update with actual mounted path

try:
    raw_data, sfreq, ch_names = load_eeg_data(gdf_path)
    print(f"Data shape: {raw_data.shape}")
    print(f"Sampling rate: {sfreq} Hz")
    print(f"Number of channels: {len(ch_names)}")
    print(f"Channel names (first 10): {ch_names[:10]}")
except FileNotFoundError:
    print("GDF file not found. Please update the gdf_path variable with a valid path.")
    # For demonstration, create synthetic data
    sfreq = 512
    n_samples = 5120  # 10 seconds
    n_channels = 64
    ch_names = [f'EEG{i:03d}' for i in range(1, n_channels+1)]
    np.random.seed(42)
    # Create synthetic EEG-like signal
    t = np.arange(n_samples) / sfreq
    raw_data = np.zeros((n_samples, n_channels))
    for ch in range(n_channels):
        # Mix of sine waves and noise
        raw_data[:, ch] = (np.sin(2*np.pi*10*t) + 0.5*np.sin(2*np.pi*1*t) + 
                          0.1*np.random.randn(n_samples))
    print("Using synthetic data for demonstration")
    print(f"Data shape: {raw_data.shape}")
    print(f"Sampling rate: {sfreq} Hz")

GDF file not found. Please update the gdf_path variable with a valid path.
Using synthetic data for demonstration
Data shape: (5120, 64)
Sampling rate: 512 Hz


In [ ]:
# Apply bandpass filter
filtered_data = apply_bandpass_filter(raw_data, sfreq, low_freq=0.1, high_freq=20.0)
print("Filtering applied successfully")

In [ ]:
# Plot before and after filtering
# Select a few representative channels and a short time window
channels_to_plot = [0, 10, 20]  # EEG001, EEG011, EEG021
time_window = slice(0, int(3 * sfreq))  # First 3 seconds

fig, axes = plt.subplots(len(channels_to_plot), 1, figsize=(12, 8), sharex=True)
fig.suptitle('EEG Signal Before and After Bandpass Filtering (0.1-20 Hz)', fontsize=14)

t = np.arange(time_window.stop) / sfreq

for i, ch_idx in enumerate(channels_to_plot):
    ax = axes[i]
    ax.plot(t, raw_data[time_window, ch_idx], label='Raw', alpha=0.7, color='blue')
    ax.plot(t, filtered_data[time_window, ch_idx], label='Filtered', alpha=0.7, color='red')
    ax.set_ylabel(f'{ch_names[ch_idx]}\n(μV)')
    ax.legend()
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:
1. Loading EEG data from a .gdf file using MNE
2. Applying a bandpass filter (0.1-20 Hz) using SciPy
3. Plotting the signal before and after filtering

The filter removes high-frequency noise (>20 Hz) and very low-frequency drift (<0.1 Hz), preserving the main EEG frequency bands.

For actual analysis, replace the placeholder path with a real .gdf file path.